# 07 Error analysis and visualization

**Question.** Where do the models err, and is uncertainty informative?

**Type.** Exploratory analysis of validation predictions of one chosen run (set `EXPERIMENT_ID` and, if needed, `RUN_ID`). No test data.

In [ ]:
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from ct_iqa.evaluation.intervals import to_score_scale, uncertainty_error_correlation
from ct_iqa.experiment.runs import read_predictions, select_run

EXPERIMENT_ID = "A_reference_resnet50"      # or B_ranking_aware_resnet50 / C_uncertainty_aware_resnet50
RUN_ID = None                               # required once several completed runs exist
run_dir = select_run(EXPERIMENT_ID, RUN_ID)
print(run_dir if run_dir else "NO COMPLETED RUN FOUND (PENDING)")


In [ ]:
if run_dir:
    p = read_predictions(run_dir)
    y, f = np.array(p["label_0_4"]), np.array(p["prediction_0_4"])
    r = f - y
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
    ax[0].scatter(y, f, s=10); ax[0].plot([0, 4], [0, 4], "k--"); ax[0].set(xlabel="label", ylabel="prediction", title="Prediction vs label")
    ax[1].hist(r, bins=20); ax[1].set(xlabel="residual (prediction - label)", ylabel="images", title="Residuals")
    ax[2].scatter(y, np.abs(r), s=10); ax[2].set(xlabel="label", ylabel="|error|", title="Absolute error vs label")
    for a in ax:
        a.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

    worst = np.argsort(-np.abs(r))[:5]
    display(pd.DataFrame({"filename": [p["filename"][i] for i in worst], "label": y[worst].round(2),
                          "prediction": f[worst].round(2), "abs error": np.abs(r[worst]).round(2)}))


## Predicted uncertainty (Experiment C runs only)

In [ ]:
if run_dir and "log_var_normalized" in p:
    _, sigma = to_score_scale(p["prediction_normalized"], p["log_var_normalized"])
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(sigma, np.abs(r), s=10)
    ax.set(xlabel="predicted sigma (0-4 scale)", ylabel="|error|", title="Uncertainty vs error")
    ax.grid(alpha=0.3)
    plt.show()
    print("Spearman(sigma, |error|):", round(uncertainty_error_correlation(sigma, np.abs(r)), 4))


Limitations to keep in mind: image-level split, 100 validation images, aleatoric-only uncertainty, labels are rater averages. See [limitations and threats](../docs/limitations-and-threats.md).